# Spam Detector
## Intro
This new challenge to build a spam detection system requires many skills that can all be accomplished in python.  First there is the matter of gathering the data.  Luckily, the Apache foundation has given us a free sample of emails, this allows us to skip gathering say personal emails and spam, which may be biased and not usable for the public.  Second we have to condition the data.  Even though the data was generously given to us, the format is still just a bunch of emails.  These emails have to be reformatted and adjusted to become words that our machine can work with effectively.  Because we will be using basic ML Models, we will need to create a sparce matrix of all emails used (not sweating the order, just managing the words used and their frequency).  Third we pipe the data into easy to process data.  Finally, we model the data and analyze our results. 

## Mission: 
We want to effectively identify an email containing spam against regular emails.  I do want to establish that we do value Recall over precision.  Because philosophically, it is easier to ignore a spam email than to dig a true email out of spam.  


# Process

## Download the data

### Fetch The data
The function below reads the url of the page the data comes from and the extensions of the data files.  The function then reviews the contents of the page and collects the tgz files.

In [1]:
import os
import tarfile
import urllib

DOWNLOAD_ROOT = "http://spamassassin.apache.org/old/publiccorpus/"
HAM_URL = DOWNLOAD_ROOT + "20030228_easy_ham.tar.bz2"
SPAM_URL = DOWNLOAD_ROOT + "20030228_spam.tar.bz2"
SPAM_PATH = os.path.join("datasets", "spam")

def fetch_spam_data(spam_url=SPAM_URL, spam_path=SPAM_PATH):
    if not os.path.isdir(spam_path):
        os.makedirs(spam_path)
    for filename, url in (("ham.tar.bz2", HAM_URL), ("spam.tar.bz2", SPAM_URL)):
        path = os.path.join(spam_path, filename)
        if not os.path.isfile(path):
            urllib.request.urlretrieve(url, path)
        tar_bz2_file = tarfile.open(path)
        tar_bz2_file.extractall(path=SPAM_PATH)
        tar_bz2_file.close()

fetch_spam_data()

### We now read the data files to list objects within python

In [2]:
HAM_DIR = os.path.join(SPAM_PATH, "easy_ham")
SPAM_DIR = os.path.join(SPAM_PATH, "spam")
ham_filenames = [name for name in sorted(os.listdir(HAM_DIR)) if len(name) > 20]
spam_filenames = [name for name in sorted(os.listdir(SPAM_DIR)) if len(name) > 20]

### Mutating the emails

In [3]:
import email
import email.policy

def load_email(is_spam, filename, spam_path=SPAM_PATH):
    directory = "spam" if is_spam else "easy_ham"
    with open(os.path.join(spam_path, directory, filename), "rb") as f:
        return email.parser.BytesParser(policy=email.policy.default).parse(f)

ham = [load_email(is_spam=False, filename=name) for name in ham_filenames]
spam = [load_email(is_spam=True, filename=name) for name in spam_filenames]

### Now let's preview the data.  
It looks like the data is still not completely preparred.  The spam data still appears to be covered with html tags.  

In [4]:
print(ham[0].get_content().strip())
print('////////////////////////////////////////////////////')
print(spam[0].get_content().strip())

Date:        Wed, 21 Aug 2002 10:54:46 -0500
    From:        Chris Garrigues <cwg-dated-1030377287.06fa6d@DeepEddy.Com>
    Message-ID:  <1029945287.4797.TMDA@deepeddy.vircio.com>


  | I can't reproduce this error.

For me it is very repeatable... (like every time, without fail).

This is the debug log of the pick happening ...

18:19:03 Pick_It {exec pick +inbox -list -lbrace -lbrace -subject ftp -rbrace -rbrace} {4852-4852 -sequence mercury}
18:19:03 exec pick +inbox -list -lbrace -lbrace -subject ftp -rbrace -rbrace 4852-4852 -sequence mercury
18:19:04 Ftoc_PickMsgs {{1 hit}}
18:19:04 Marking 1 hits
18:19:04 tkerror: syntax error in expression "int ...

Note, if I run the pick command by hand ...

delta$ pick +inbox -list -lbrace -lbrace -subject ftp -rbrace -rbrace  4852-4852 -sequence mercury
1 hit

That's where the "1 hit" comes from (obviously).  The version of nmh I'm
using is ...

delta$ pick -version
pick -- nmh-1.0.4 [compiled on fuchsia.cs.mu.OZ.AU at Sun Mar 17 14:55:56 

Lets look at another set.  The emails don't look too supprizing on what is good and spam.  It appears that the first email appears to be pleant of information passed on with a link included.  The spam appears to be claims(too good to be true) and links.  

In [5]:
print(ham[3].get_content().strip())
print('////////////////////////////////////////////////////')
print(spam[3].get_content().strip())

Klez: The Virus That Won't Die
 
Already the most prolific virus ever, Klez continues to wreak havoc.

Andrew Brandt
>>From the September 2002 issue of PC World magazine
Posted Thursday, August 01, 2002


The Klez worm is approaching its seventh month of wriggling across 
the Web, making it one of the most persistent viruses ever. And 
experts warn that it may be a harbinger of new viruses that use a 
combination of pernicious approaches to go from PC to PC.

Antivirus software makers Symantec and McAfee both report more than 
2000 new infections daily, with no sign of letup at press time. The 
British security firm MessageLabs estimates that 1 in every 300 
e-mail messages holds a variation of the Klez virus, and says that 
Klez has already surpassed last summer's SirCam as the most prolific 
virus ever.

And some newer Klez variants aren't merely nuisances--they can carry 
other viruses in them that corrupt your data.

...

http://www.pcworld.com/news/article/0,aid,103259,00.asp
____

## Divide the datasets into training and test

### Adds a dummy identifier for dataset origin

In [6]:
import numpy as np
y = np.array([0] * len(ham) + [1] * len(spam))

### Divide the datasets
These simple commands shuffle the ham and spam together into a test set alongside the dummy indicator.  We are giving the data set a 80/20 split for train/testing.  

In [7]:
import numpy as np
from sklearn.model_selection import train_test_split 
X = np.array(ham + spam, dtype=object)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Parsing the email (HTML Handling)
  Because some of the emails are reply or forarded email, we want to assess what is a multipart email.  The first condition identifes that it is in fact an email.  If so, we have nothing further to process.  Otherwise, we look at the email and see if that is the last piece, if so, we get the email otherwise, we run the function again recursively. 

In [8]:
def get_email_structure(email):
    if isinstance(email, str):
        return email
    payload = email.get_payload()
    if isinstance(payload, list):
        return "multipart({})".format(", ".join([
            get_email_structure(sub_email)
            for sub_email in payload
        ]))
    else:
        return email.get_content_type()

The below function handles all of the seen HTML code.  The first line identifies that the html item is a head and is therefore dropped.  The second line converts all of the hyperlinks into the test 'hyperlink'.  The last lines convert the text into pure text.

In [9]:
import re
from html import unescape
from collections import Counter

def html_to_plain_text(html):
    text = re.sub('<head.*?>.*?</head>', '', html, flags=re.M | re.S | re.I)
    text = re.sub('<a\s.*?>', ' HYPERLINK ', text, flags=re.M | re.S | re.I)
    text = re.sub('<.*?>', '', text, flags=re.M | re.S)
    text = re.sub(r'(\s*\n)+', '\n', text, flags=re.M | re.S)
    return unescape(text)





The function below converts an email to pure text.  There may still be html tags in the content.  If so we strip them and return only text. 

In [10]:
def email_to_text(e):
    markup = None
    for part in e.walk():
        ctype = part.get_content_type()
        if not ctype in ("text/plain", "text/html"):
            continue
        try:
            content = part.get_content()
        except:
            content = str(part.get_payload())
        if ctype == "text/plain":
            return content
        else:
            markup = content
    if markup:
        return html_to_plain_text(markup)

Below we import the urextractor.  This function will capture the urls in the text.  

In [11]:
import urlextract # may require an Internet connection to download root domain names
url_extractor = urlextract.URLExtract()

It wouldn’t be a NLP project if we were not importing the nltk package.   The stemmer will unite our plurals and our singles giving these words much more significance instead of splitting these words into there plurality.  

In [12]:
import nltk
stemmer = nltk.PorterStemmer()

# Creating the Sparse Matrix
Here we will be able to create data that can be fed into our models that will count the number of instances each word is used.  The final product will consist of an array that represents words and their frequency.  

The below code is what is able to create the array of the words used and the instances that are used.  The true object appears to an array of emails that are manipulated to strip headers, be all lower case, replace urls and numbers.  The primary function of the class (.transform) is to output an array of found strings.  
This is done by 1 initializing the object: we want all of the letters consistant (lowercase) not polluted with special characters.  2  Fetching all of the URLs - we may be able to catch good and bad ones.  3 Replacing numbers with a constant text - numbers will be common, but when they are distributed among thousands, we will not see them as significant.  

In [13]:
from sklearn.base import BaseEstimator, TransformerMixin

class EmailToWordCounterTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, strip_headers=True, lower_case=True, remove_punctuation=True,
                 replace_urls=True, replace_numbers=True, stemming=True):
        self.strip_headers = strip_headers
        self.lower_case = lower_case
        self.remove_punctuation = remove_punctuation
        self.replace_urls = replace_urls
        self.replace_numbers = replace_numbers
        self.stemming = stemming
    def fit(self, X, y=None):
        return self
    def transform(self, X, y=None):
        X_transformed = []
        for email in X:
            text = email_to_text(email) or ""
            if self.lower_case:
                text = text.lower()
            if self.replace_urls and url_extractor is not None:
                urls = list(set(url_extractor.find_urls(text)))
                urls.sort(key=lambda url: len(url), reverse=True)
                for url in urls:
                    text = text.replace(url, " URL ")
            if self.replace_numbers:
                text = re.sub(r'\d+(?:\.\d*)?(?:[eE][+-]?\d+)?', 'NUMBER', text)
            if self.remove_punctuation:
                text = re.sub(r'\W+', ' ', text, flags=re.M)
            word_counts = Counter(text.split())
            if self.stemming and stemmer is not None:
                stemmed_word_counts = Counter()
                for word, count in word_counts.items():
                    stemmed_word = stemmer.stem(word)
                    stemmed_word_counts[stemmed_word] += count
                word_counts = stemmed_word_counts
            X_transformed.append(word_counts)
        return np.array(X_transformed)

Below, we hardcoded the vocabulary size to 1000 that is, we are looking for the number of items that exist in the training set and see how the intersect with any given email.  Rows, columns and data are all initialized as null lists and we iterate each row into these lists and package them into a single csr_matrix.

In [14]:
from scipy.sparse import csr_matrix

class WordCounterToVectorTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, vocabulary_size=1000):
        self.vocabulary_size = vocabulary_size
    def fit(self, X, y=None):
        total_count = Counter()
        for word_count in X:
            for word, count in word_count.items():
                total_count[word] += min(count, 10)
        most_common = total_count.most_common()[:self.vocabulary_size]
        self.vocabulary_ = {word: index + 1 for index, (word, count) in enumerate(most_common)}
        return self
    def transform(self, X, y=None):
        rows = []
        cols = []
        data = []
        for row, word_count in enumerate(X):
            for word, count in word_count.items():
                rows.append(row)
                cols.append(self.vocabulary_.get(word, 0))
                data.append(count)
        return csr_matrix((data, (rows, cols)), shape=(len(X), self.vocabulary_size + 1))
    

## Pipeline
The below Pipeline packages the two above objects into a single new list that includes the final sparse matrix.  This pipeline can capture both object functions.  The result is in fact a sparse matrix type object with many rows and columns.      

In [15]:
from sklearn.pipeline import Pipeline

preprocess_pipeline = Pipeline([
    ("email_to_wordcount", EmailToWordCounterTransformer()),
    ("wordcount_to_vector", WordCounterToVectorTransformer()),
])

X_train_transformed = preprocess_pipeline.fit_transform(X_train)

# Modeling 
At last, it is time to build the model that we have been working towards.  We will build a logistic model, a decision tree, random forest and k-Nearest Neighbors. For ease of use and resuability, I have created two functions (so I can run code in two cells) to show off the cross_val_score and take a dive into the confusion analysis.  Using this function, anyone who develops a competing model can analyze the score and confusion outlook

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
def model_analysis(clf): 
    score = cross_val_score(clf, X_train_transformed, y_train, cv=3, verbose=3)
    print("{:.2f}%".format(100*score.mean()))

In [17]:
from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
def model_analysis2(clf): 
    X_test_transformed = preprocess_pipeline.transform(X_test)
    clf.fit(X_train_transformed, y_train)
    y_pred = clf.predict(X_test_transformed)
    print("F1: {:.2f}%".format(100 * f1_score(y_test, y_pred)))
    print("Precision: {:.2f}%".format(100 * precision_score(y_test, y_pred)))
    print("Recall: {:.2f}%".format(100 * recall_score(y_test, y_pred)))
    print(confusion_matrix(y_test, y_pred))

## Logistic Regression
If you see below, using a Logistic Regression Model to solve for the problem gives us 98.5% accuracy.

In [18]:
from sklearn.linear_model import LogisticRegression
log_clf = LogisticRegression(max_iter=1000, random_state=42)
model_analysis(log_clf)

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.


[CV] END ................................ score: (test=0.981) total time=   0.3s


[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.3s remaining:    0.0s


[CV] END ................................ score: (test=0.984) total time=   0.3s


[Parallel(n_jobs=1)]: Done   2 out of   2 | elapsed:    0.7s remaining:    0.0s


[CV] END ................................ score: (test=0.990) total time=   0.4s
98.50%


[Parallel(n_jobs=1)]: Done   3 out of   3 | elapsed:    1.3s finished


It looks like our Precision is 95.88% and our Recall 97.89%.  That is we let spam into our inbox less than 5% of the time and require users to dig emails out of SPAM less than 3% of the time.  This is in my philosiphy ideal, because it is easier to ignore emails than to recover emails - especially if you don't know they are there.


In [19]:
model_analysis2(log_clf)

F1: 96.88%
Precision: 95.88%
Recall: 97.89%
[[501   4]
 [  2  93]]


## Decision Tree 
As you can see below, the decision tree has a classification rate of 95% (give or take random state), certianly lower than the logistic regression, but certianly nothing to weep about.  Unless there is a substancially great Recall in this model, we will not be using it.  If we were to continue this analysis beyond the scope of the project, this would be one of the best models for explaining the data to 1) build better models and 2) explain/advertize the policy of the model.  

In [33]:
import random 
np.random.seed(42)
from sklearn import tree
tree_clf = tree.DecisionTreeClassifier(splitter = 'random')
model_analysis(tree_clf)

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s remaining:    0.0s


[CV] END ................................ score: (test=0.950) total time=   0.0s
[CV] END ................................ score: (test=0.946) total time=   0.0s
[CV] END ................................ score: (test=0.946) total time=   0.0s
94.75%


[Parallel(n_jobs=1)]: Done   2 out of   2 | elapsed:    0.1s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   3 out of   3 | elapsed:    0.2s finished


Interestingly, the scores below, give us very dissapointing results.  The cross validation rate appears to be well below the relatively promising 

In [21]:
model_analysis2(tree_clf)

F1: 85.42%
Precision: 84.54%
Recall: 86.32%
[[490  15]
 [ 13  82]]


# Random Forest
We have made a highly competitive model now.  Using something a little more cutting edge (but harder to explain), we have improved the tree model to produce something that can better represent binary response.  

In [35]:
from sklearn.ensemble import RandomForestClassifier
rf_clf = RandomForestClassifier(max_depth=60, random_state=0)
model_analysis(rf_clf)

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.


[CV] END ................................ score: (test=0.978) total time=   0.5s


[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.5s remaining:    0.0s


[CV] END ................................ score: (test=0.980) total time=   0.6s


[Parallel(n_jobs=1)]: Done   2 out of   2 | elapsed:    1.2s remaining:    0.0s


[CV] END ................................ score: (test=0.983) total time=   0.5s
98.00%


[Parallel(n_jobs=1)]: Done   3 out of   3 | elapsed:    1.9s finished


It does look like even though the the model was outstanding, the confusion matrix seems to weigh towards blocking emails non-spam emails at the expense of correctly blocking spam emails.  Given my previously mentioned philosiphy, I would still turn towards the logistic model because it is more in lign of what we are looking for - if the philosiphy changes, I will swiftly declare this one the winner over logistic.  

In [36]:
model_analysis2(rf_clf)

F1: 94.51%
Precision: 98.85%
Recall: 90.53%
[[504   1]
 [  9  86]]


# Nearest K
Again, not a superb model, but we have seen before looking under the hood may give us alternative measures to making the decision for the model.

In [24]:
from sklearn.neighbors import KNeighborsClassifier
neig_clf = KNeighborsClassifier(n_neighbors=3)
model_analysis(neig_clf)

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.


[CV] END ................................ score: (test=0.935) total time=   0.2s
[CV] END ................................ score: (test=0.941) total time=   0.0s


[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.2s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   2 out of   2 | elapsed:    0.3s remaining:    0.0s


[CV] END ................................ score: (test=0.917) total time=   0.0s
93.12%


[Parallel(n_jobs=1)]: Done   3 out of   3 | elapsed:    0.4s finished


We see now that this model has the absolute worst Recall.  This model will not be further considered.  

In [25]:
model_analysis2(neig_clf)

F1: 83.43%
Precision: 91.25%
Recall: 76.84%
[[498   7]
 [ 22  73]]


# Conclusion.  
As you can see, there are multiple ways to model the classification of spam/not spam in an email.  From the above experiment, it appears that the best way to do this is with a logistic regression- a favorite for binary classifications.  As spam filters are implemented senders of spam will continue to persist to bypass the spam filter we should keep all models handy (possibly dropping knn and Tree models) for changing data considering alternative models, which may be more resilient against more ingenious spam messages in a game of ‘cat and mouse’.  
Additionally, as individual email and spam volume build up, it may be necessary to build custom made email filters that adjust to a user’s needs – perhaps they want a filter that’s more liberal or conservative as well as a spam definition of their own based on flagged emails.  
